In [2]:
import os
import json
import numpy as np
from scipy.signal import find_peaks

INPUT_DIR = "../data/interpolated_json"
OUTPUT_DIR = "../data/labeled_json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Looking inside:", os.path.abspath(INPUT_DIR))
print("Folders found:", os.listdir(INPUT_DIR))

def extract_signal(frames, exercise):
    signal = []

    for frame in frames:
        lm = frame["landmarks"]

        if exercise == "bench_press":
            value = (lm[15][1] + lm[16][1]) / 2   # wrists

        elif exercise == "pull_up":
            value = (lm[11][1] + lm[12][1]) / 2   # shoulders

        elif exercise == "push_up":
            value = (lm[11][1] + lm[12][1]) / 2   # shoulders

        elif exercise == "squat":
            value = (lm[11][1] + lm[12][1]) / 2   # shoulders/body movement

        else:
            value = lm[12][1]

        signal.append(value)

    return np.array(signal)


def detect_peaks(signal, exercise):
    signal = np.array(signal)

    # smooth signal
    window = 7
    if len(signal) >= window:
        signal = np.convolve(signal, np.ones(window) / window, mode="same")

    params = {
        "bench_press": {"prominence": 0.08, "distance": 20},
        "pull_up": {"prominence": 0.05, "distance": 20},
        "push_up": {"prominence": 0.06, "distance": 20},
        "squat": {"prominence": 0.07, "distance": 25},
    }

    p = params.get(exercise, {"prominence": 0.08, "distance": 20})

    peaks, _ = find_peaks(signal, prominence=p["prominence"], distance=p["distance"])
    valleys, _ = find_peaks(-signal, prominence=p["prominence"], distance=p["distance"])

    chosen = peaks if len(peaks) >= len(valleys) else valleys

    return chosen.tolist()


total_saved = 0

for exercise in os.listdir(INPUT_DIR):
    exercise_path = os.path.join(INPUT_DIR, exercise)

    if not os.path.isdir(exercise_path):
        continue

    save_exercise_path = os.path.join(OUTPUT_DIR, exercise)
    os.makedirs(save_exercise_path, exist_ok=True)

    files = [f for f in os.listdir(exercise_path) if f.endswith(".json")]

    print(f"\nProcessing {exercise}")
    print("JSON files found:", len(files))

    saved_count = 0

    for file in files:
        file_path = os.path.join(exercise_path, file)

        try:
            with open(file_path, "r") as f:
                frames = json.load(f)

            signal = extract_signal(frames, exercise)
            peaks = detect_peaks(signal, exercise)

            output = {
                "frames": frames,
                "time_points": peaks,
                "rep_count": len(peaks)
            }

            save_path = os.path.join(save_exercise_path, file)

            with open(save_path, "w") as f:
                json.dump(output, f)

            saved_count += 1
            total_saved += 1

        except Exception as e:
            print(f"Skipping {file}: {e}")

    print("Saved:", saved_count)

print("\nTOTAL SAVED:", total_saved)
print("Output folder:", os.path.abspath(OUTPUT_DIR))

Looking inside: c:\Users\marie\OneDrive\Documents\GitHub\CoreSet-AI-Fitness-Tracker-ML-Pipeline\data\interpolated_json
Folders found: ['bench_press', 'pull_up', 'push_up', 'squat']

Processing bench_press
JSON files found: 250
Saved: 250

Processing pull_up
JSON files found: 250
Saved: 250

Processing push_up
JSON files found: 250
Saved: 250

Processing squat
JSON files found: 250
Saved: 250

TOTAL SAVED: 1000
Output folder: c:\Users\marie\OneDrive\Documents\GitHub\CoreSet-AI-Fitness-Tracker-ML-Pipeline\data\labeled_json
